In [1]:
import pandas as pd
from scipy.stats import linregress
import numpy as np

def auditar_trafego_fulltime(csv_entrada, carros_alvo=[10, 80], limite_ar_sujo=0.4):
    print(f"Auditoria de Tráfego Direcionada: Full Time Sports (Carros {carros_alvo})")
    print(f"Régua de sensibilidade aerodinâmica: Perdas maiores que {limite_ar_sujo}s")
    print("-" * 85)

    df = pd.read_csv(csv_entrada, sep=';', decimal=',')
    df['Volta_Absoluta'] = df.groupby(['Carro', 'Stint']).cumcount() + 1
    
    # Filtrar apenas os nossos pilotos
    df_alvos = df[df['Carro'].isin(carros_alvo)]

    for carro in carros_alvo:
        print(f"\nANALISANDO CARRO {carro}")
        df_carro = df_alvos[df_alvos['Carro'] == carro]
        
        for stint, grupo in df_carro.groupby('Stint'):
            # 1. A Régua (A Volta Perfeita)
            voltas_limpas = grupo[(grupo['Tipo_Volta'] == 'Push') & (grupo['Outlier'] == False)]
            
            # Vamos testar TODAS as voltas voadoras, mesmo as que o ETL não achou que era Outlier
            voltas_voadoras = grupo[~grupo['Tipo_Volta'].isin(['In', 'Out'])]
            
            if len(voltas_limpas) >= 3:
                x_limpo = voltas_limpas['Volta_Absoluta'].values
                y_limpo = voltas_limpas['Lap Tm (Segundos)'].values
                slope, intercept, _, _, _ = linregress(x_limpo, y_limpo)
                
                perdas_sutis = []
                perdas_pesadas = []
                
                # 2. Avaliando cada volta do Stint
                for _, volta in voltas_voadoras.iterrows():
                    num_volta = volta['Volta_Absoluta']
                    tempo_real = volta['Lap Tm (Segundos)']
                    tempo_ideal = (slope * num_volta) + intercept
                    delta_perda = tempo_real - tempo_ideal
                    
                    # Classificação do Tráfego
                    if limite_ar_sujo <= delta_perda < 1.5:
                        perdas_sutis.append(delta_perda) # Ar Sujo / Esteira Aerodinâmica
                    elif 1.5 <= delta_perda <= 5.0:
                        perdas_pesadas.append(delta_perda) # Tráfego Pesado / Briga por Posição
                
                # 3. Imprimindo o Dossiê do Stint
                total_voltas = len(voltas_voadoras)
                afetadas = len(perdas_sutis) + len(perdas_pesadas)
                
                if afetadas > 0:
                    custo_total = sum(perdas_sutis) + sum(perdas_pesadas)
                    print(f"  ↳ Stint {stint} ({total_voltas} voltas): {afetadas} voltas no tráfego. Custo total: +{round(custo_total, 3)}s jogados fora.")
                    if perdas_sutis:
                        print(f"      - Ar Sujo ({len(perdas_sutis)}v): Média de perda = {round(np.mean(perdas_sutis), 3)}s/volta")
                    if perdas_pesadas:
                        print(f"      - Tráfego Pesado ({len(perdas_pesadas)}v): Média de perda = {round(np.mean(perdas_pesadas), 3)}s/volta")
                else:
                    print(f"  ↳ Stint {stint} ({total_voltas} voltas): Stint perfeito. Correu de cara pro vento (Ar Limpo).")
            else:
                print(f"  ↳ Stint {stint}: Dados insuficientes para traçar a reta ideal.")
    print("-" * 85)

# --- ÁREA DE EXECUÇÃO ---
arquivo_corrida = '../data/03_processed/TELEMETRIA_ESTRATEGIA_T2.csv'
auditar_trafego_fulltime(arquivo_corrida, carros_alvo=[10, 80], limite_ar_sujo=0.4)

Auditoria de Tráfego Direcionada: Full Time Sports (Carros [10, 80])
Régua de sensibilidade aerodinâmica: Perdas maiores que 0.4s
-------------------------------------------------------------------------------------

ANALISANDO CARRO 10
  ↳ Stint 1 (7 voltas): 1 voltas no tráfego. Custo total: +2.681s jogados fora.
      - Tráfego Pesado (1v): Média de perda = 2.681s/volta
  ↳ Stint 2 (7 voltas): 1 voltas no tráfego. Custo total: +1.347s jogados fora.
      - Ar Sujo (1v): Média de perda = 1.347s/volta
  ↳ Stint 3 (4 voltas): 1 voltas no tráfego. Custo total: +0.902s jogados fora.
      - Ar Sujo (1v): Média de perda = 0.902s/volta

ANALISANDO CARRO 80
  ↳ Stint 1 (8 voltas): 3 voltas no tráfego. Custo total: +5.992s jogados fora.
      - Ar Sujo (2v): Média de perda = 1.402s/volta
      - Tráfego Pesado (1v): Média de perda = 3.188s/volta
  ↳ Stint 2 (6 voltas): 2 voltas no tráfego. Custo total: +3.661s jogados fora.
      - Ar Sujo (1v): Média de perda = 1.439s/volta
      - Tráfego 